# PerfumeInsightLab
## Notebook 01 : Data Overview
--------------------------------------------------------------------  
#### Part of the multi-notebook EDA workflow : 01 → 02 (Thematic EDA) → 03 (Quantitative EDA) → 04 (Storytelling & Insights)
#### 1. Import librairies
#### 2. Load dataset
#### 3. Quick overview
#### 4. Check missing values
#### 5. Check duplicates
#### 6. Final data cleaning / Consolidation


In [ ]:
# ------------------------------------------------------------------
# 1. Import libraries
# ------------------------------------------------------------------
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import unidecode

# ------------------------------------------------------------------
# 2. Load dataset
# ------------------------------------------------------------------
df = pd.read_csv("../data/fra_cleaned.csv", encoding="ISO-8859-1", sep=";")

In [ ]:
# ------------------------------------------------------------------
# 3. Quick overview
# ------------------------------------------------------------------
df.shape

In [ ]:
display(df.head())         

In [ ]:
print("DATA INFO :")            # Info about data types and non-null values
df.info()

In [ ]:
print("DESCRIPTIVE STATISTICS :")              # Descriptive statistics (numerical + categorical)
display(df.describe(include='all'))

In [ ]:
# ------------------------------------------------------------------
# 4. Check missing values
# ------------------------------------------------------------------
missing = df.isna().sum()
print("MISSING VALUES PER COLUMN :\n", missing)

In [ ]:
missing[missing > 1000]               # Check which columns have a large nber of missing values

In [ ]:
duplicates = df[df.duplicated()]      # Display duplicated rows to inspect potential data repetition
duplicates.head()

In [ ]:
df.nunique()                          # Count the nber of unique values in each column

In [ ]:
df['Brand'].value_counts().head(10)          # Show the most frequent values for main categorical columns
df['Country'].value_counts().head(10)
df['Gender'].value_counts()

In [ ]:
df['Rating Value'].hist(bins=10)             # Visualize the distribution of ratings

In [ ]:
df.corr(numeric_only=True)              # Compute correlations only for numeric columns

#### Interpretation of Correlation Matrix

The correlation coefficient between 'Year' and 'Rating Count' is **-0.0994**

This value, being close to 0, indicates an **extremely weak inverse linear relationship**.  
While older perfumes (lower 'Year') tend to have slightly higher vote counts, the year of release is a very poor predictor of a perfume's overall popularity (Rating Count).   
This finding statistically supports the necessity of using the **Weighted Rating** to correct for the popularity bias.

In [ ]:
# ------------------------------------------------------------------
# 5. Check duplicates
# ------------------------------------------------------------------
duplicates = df.duplicated().sum()
print("NUMBER OF DUPLICATED ROWS :", duplicates)

In [ ]:
df.dtypes
df.isna().sum().sort_values(ascending=False)
df['Gender'].value_counts(normalize=True).mul(100).round(4)

####  Gender Distribution Analysis

This cell confirms the current market distribution by gender, based on volume :  
**Women's fragrances** dominate the volume with **~ 47.28%**   
**Unisex fragrances** represent a significant market share at **~ 32.00%**, demonstrating the industry's shift towards gender-neutral products.   
**Men's fragrances** hold the smallest segment at **~ 20.73%**   

In [ ]:
df['Brand'].value_counts().head(10)
df.duplicated(subset=['Perfume','Brand']).sum()

In [ ]:
duplicates_to_check = df[df.duplicated(subset=['Perfume', 'Brand'], keep=False)]
display(duplicates_to_check.sort_values(by=['Perfume', 'Brand']).head(10))

#### Data Integrity : 

#### Duplicate Validation
I performed 2 levels of duplicate checks:
1. **Full-Row Duplicates :** `df.duplicated().sum()` returned **0**, meaning no two rows are 100% identical across all metadata.  
2. **Business Duplicates :** Using a subset (`Perfume`, `Brand`, `Year`), I identified **217 redundancies**. These are entries representing the same product version but with slight variations in secondary data (like rating counts or note descriptions).

#### Brand Distribution and Duplicate Check
**Top Brands :** The frequency distribution reveals which brands dominate the dataset in terms of volume.  
**Duplicate Detection :** The system identified **217 duplicate entries** based on the 'Perfume' and 'Brand' combination.  

#### Duplicates vs. Flankers
In perfumery, **flankers** (variations of an original scent) are essential data points.    
To avoid deleting legitimate flankers, the duplicate removal process was carefully configured :   
**Legitimate Flankers :** Preserved, as they typically have distinct names (e.g., "EDP" vs. "Intense") or different launch years.  
**Technical Duplicates :** 217 entries sharing the exact same Perfume Name, Brand and Year were removed.   
These represent scraping redundancies or identical re-entries that would otherwise bias our statistical averages.

**→ These 217 duplicates have been removed to ensure each unique fragrance is counted only once in the market analysis.**

In [ ]:
df['Rating Value'] = pd.to_numeric(df['Rating Value'], errors='coerce')
df[['Rating Value','Rating Count']].corr().round(2)

#### Correlation : Rating Value vs. Rating Count

The correlation coefficient between **Rating Value** and **Rating Count** is **0.02**.

**Result :** There is **no linear correlation** between how highly a perfume is rated and its popularity (nber of votes).  
**Insight :** Popularity does not imply quality. A high volume of ratings does not systematically result in a higher or lower score.  
**Conclusion :** This statistical independence justifies the use of a **Weighted Rating** to identify truly top-rated fragrances while filtering out low-volume outliers.   

In [ ]:
# ------------------------------------------------------------------
# 6. Final Data Cleaning / Consolidation
# ------------------------------------------------------------------
if 'Year' in df.columns:                                                             # 1. Convert 'Year' to integer and handle missing values
    df['Year'] = pd.to_numeric(df['Year'], errors='coerce').fillna(0).astype(int)

df = df.rename(columns={                                                             # 2. Rename note columns for clarity
    'Top': 'Top Notes',
    'Middle': 'Middle Notes',
    'Base': 'Base Notes'
})

if 'url' in df.columns:                                                              # 3. Drop 'url' column if it exists
    df = df.drop(columns=['url'])

if 'Perfumer1' in df.columns:                                                        # 4. Merge Perfumer1 and Perfumer2 safely
    df['Perfumer'] = df['Perfumer1']
    if 'Perfumer2' in df.columns:
        # Combine if Perfumer2 is not NaN
        df['Perfumer'] = np.where(df['Perfumer2'].notna(),
                                   df['Perfumer1'].astype(str) + ', ' + df['Perfumer2'].astype(str),
                                   df['Perfumer1'])
    df = df.drop(columns=['Perfumer1', 'Perfumer2'], errors='ignore')        # Remove original columns after merging


accord_cols = ['mainaccord2', 'mainaccord3', 'mainaccord4', 'mainaccord5']           # 5. Fill missing mainaccord values
for col in accord_cols:
    if col in df.columns:
        df[col] = df[col].fillna('unknown')

if 'Rating Value' in df.columns:                                                     # 6. Standardize Rating Value (float) and Rating Count (int)
    df['Rating Value'] = (df['Rating Value'].astype(str).str.replace(',', '.', regex=False).astype(float))

if 'Rating Count' in df.columns:
    df['Rating Count'] = (
        df['Rating Count'].astype(str)
        .str.replace(',', '', regex=False)           # Remove thousands separator
    )
    df['Rating Count'] = pd.to_numeric(df['Rating Count'], errors='coerce').fillna(0).astype(int)
                                                                                    
df = df.drop_duplicates(subset=['Perfume', 'Brand', 'Year'], keep='first')          # 7. Remove technical duplicates (same Perfume, Brand, and Year)

df.to_csv("../data/fra_cleaned_v2.csv", index=False, sep=";")                       # 8. Save cleaned dataset
print("SUCCESS ! Cleaned dataset saved as 'fra_cleaned_v2.csv'")

display(df.head())                                                                  # Final Inspection

#### Summary

| Metric | Observation |
|:-------|:------------|
| **Initial Rows** | 22 380 |
| **Columns After Cleaning** | 16 |
| **Missing Values** | Handled (imputed or marked as 'unknown') |
| **Year Range** | 1780 – 2024 |
| **Gender Distribution** | Women : 47%, Unisex : 32%, Men : 21% |
| **Duplicates Removed** | 217 technical duplicates (Brand + Perfume + Year) |
| **Cleaned Dataset** | Exported as `fra_cleaned_v2.csv` |

#### Key Findings & Cleaning Steps
* **Data Integrity :** Distinguished between technical duplicates and industry "flankers" (reformulations) by including the launch year in the duplicate check.
* **Feature Engineering :** Unified `Perfumer1` and `Perfumer2` into a single `Perfumer` field and renamed olfactory notes for better readability.
* **Correlation Insights :** Confirmed that Rating Value and Rating Count are independent ($\rho = 0.02$), justifying the future use of a Weighted Rating.
* **Normalization :** Standardized text fields (casing, whitespace) and converted numerical strings to robust `float` and `int` types.

#### The dataset is now statistically sound and ready for deep-dive analysis